<a href="https://colab.research.google.com/github/Ahowardjr/MSSP6070/blob/main/Assignments/Assignment%202/Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
from pathlib import Path
from IPython.display import display

# Load the dataset with latin1 encoding and low_memory=False to suppress mixed type warnings
df = pd.read_csv('/content/InternetSales.csv', encoding='latin1', low_memory=False)
display(df.head())

,ID,CustomerKey,GeographyKey,City,CountryRegionCode,EnglishCountryRegionName,CustomerAlternateKey,Title,FirstName,MiddleName,...,ProductStandardCost,TotalProductCost,SalesAmount,TaxAmt,Freight,CarrierTrackingNumber,CustomerPONumber,OrderDate,DueDate,ShipDate
0,0,11000,26,Rockhampton,AU,Australia,AW00011000,NaN,Jon,V,...,1912.1544,1912.1544,3399.99,271.9992,84.9998,NaN,NaN,1/19/2011,1/31/2011,1/26/2011
1,1,11000,26,Rockhampton,AU,Australia,AW00011000,NaN,Jon,V,...,1265.6195,1265.6195,2319.99,185.5992,57.9998,NaN,NaN,1/18/2013,1/30/2013,1/25/2013
2,2,11000,26,Rockhampton,AU,Australia,AW00011000,NaN,Jon,V,...,8.2205,8.2205,21.98,1.7584,0.5495,NaN,NaN,1/18/2013,1/30/2013,1/25/2013
3,3,11000,26,Rockhampton,AU,Australia,AW00011000,NaN,Jon,V,...,1481.9379,1481.9379,2384.07,190.7256,59.6018,NaN,NaN,5/3/2013,5/15/2013,5/10/2013
4,4,11000,26,Rockhampton,AU,Australia,AW00011000,NaN,Jon,V,...,10.8423,10.8423,28.99,2.3192,0.7248,NaN,NaN,5/3/2013,5/15/2013,5/10/2013


In [ ]:
df.columns.tolist()

['ID',
 'CustomerKey',
 'GeographyKey',
 'City',
 'CountryRegionCode',
 'EnglishCountryRegionName',
 'CustomerAlternateKey',
 'Title',
 'FirstName',
 'MiddleName',
 'LastName',
 'NameStyle',
 'BirthDate',
 'MaritalStatus',
 'Suffix',
 'Gender',
 'EmailAddress',
 'YearlyIncome',
 'TotalChildren',
 'NumberChildrenAtHome',
 'EnglishEducation',
 'SpanishEducation',
 'FrenchEducation',
 'EnglishOccupation',
 'SpanishOccupation',
 'FrenchOccupation',
 'HouseOwnerFlag',
 'NumberCarsOwned',
 'AddressLine1',
 'AddressLine2',
 'Phone',
 'DateFirstPurchase',
 'CommuteDistance',
 'ProductKey',
 'EnglishProductName',
 'OrderDateKey',
 'DueDateKey',
 'ShipDateKey',
 'CustomerKey.1',
 'PromotionKey',
 'CurrencyKey',
 'SalesTerritoryKey',
 'SalesOrderNumber',
 'SalesOrderLineNumber',
 'RevisionNumber',
 'OrderQuantity',
 'UnitPrice',
 'ExtendedAmount',
 'UnitPriceDiscountPct',
 'DiscountAmount',
 'ProductStandardCost',
 'TotalProductCost',
 'SalesAmount',
 'TaxAmt',
 'Freight',
 'CarrierTrackingNumber

Internet Sales Summary by Year
To see the growth trajectory of Adventure Works, let's extract the year from `OrderDate` and calculate total sales and order counts per year.

In [ ]:
# Convert OrderDate to datetime
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
df['OrderYear'] = df['OrderDate'].dt.year

# Group sales by Year
yearly_sales = df.groupby('OrderYear').agg(
    Total_Sales=('SalesAmount', 'sum'),
    Order_Count=('SalesOrderNumber', 'nunique') if 'SalesOrderNumber' in df.columns else ('ID', 'count')
).reset_index()

# Calculate YoY Growth in Sales
yearly_sales['YoY_Growth_Pct'] = yearly_sales['Total_Sales'].pct_change() * 100

print("--- Yearly Internet Sales Summary ---")
display(yearly_sales.style.format({
    'Total_Sales': '${:,.2f}',
    'Order_Count': '{:,}',
    'YoY_Growth_Pct': '{:.2f}%'
}))

--- Yearly Internet Sales Summary ---


,OrderYear,Total_Sales,Order_Count,YoY_Growth_Pct
0,2010,"$43,421.04",14,nan%
1,2011,"$7,075,525.93","2,216",16195.16%
2,2012,"$5,842,485.20","3,269",-17.43%
3,2013,"$16,351,550.34","21,289",179.87%
4,2014,"$45,694.72",871,-99.72%


In [ ]:
total_sales = df["SalesAmount"].sum()

print(f"Total Internet Sales: ${total_sales:,.2f}")

Total Internet Sales: $29,358,677.22


In [ ]:
yearly_country_sales = df.groupby(
    ['OrderYear', 'EnglishCountryRegionName']
)['SalesAmount'].sum().reset_index(name='Total_Sales')

display(yearly_country_sales.style.format({
    'Total_Sales': '{:,.0f}'}))

,OrderYear,EnglishCountryRegionName,Total_Sales
0,2010,Australia,"20,910"
1,2010,Canada,"3,578"
2,2010,France,"3,400"
3,2010,United Kingdom,699
4,2010,United States,"14,834"
5,2011,Australia,"2,563,732"
6,2011,Canada,"571,572"
7,2011,France,"410,845"
8,2011,Germany,"520,500"
9,2011,United Kingdom,"550,591"


Generate summary statistics including min, max, and standard deviation for Yearly Income, Gender by Country Regions

In [ ]:
income_summary = df.groupby(
    ['EnglishCountryRegionName', 'Gender']
)['YearlyIncome'].agg(['count', 'mean', 'min', 'max', 'std'])

display(income_summary.style.format('{:,.2f}'))

What is the average price of the product category that generates the most sales?


In [ ]:
results = (
    df.assign(
        Profit=lambda x: x['SalesAmount'] - x['TotalProductCost']
    )
    .groupby("EnglishProductName")
    .agg({
        'OrderQuantity': 'sum',
        'UnitPrice': 'mean',
        'SalesAmount': 'sum',
        'TotalProductCost': 'sum',
        'Profit': 'sum'
    })
    .reset_index()
    .sort_values(by='SalesAmount', ascending=False)
    .head(10)
)

display(results.style.format({
    'OrderQuantity': '{:,.0f}',
    'UnitPrice': '{:,.2f}',
    'SalesAmount': '{:,.2f}',
    'TotalProductCost': '{:,.2f}',
    'Profit': '{:,.2f}'
}))

print(f"The product with the most sales is {top_product['EnglishProductName']}, with an average price of ${top_product['UnitPrice']:,.2f}.")

,EnglishProductName,OrderQuantity,UnitPrice,SalesAmount,TotalProductCost,Profit
34,"Mountain-200 Black, 46",620,"2,215.27","1,373,469.55","746,847.97","626,621.57"
33,"Mountain-200 Black, 42",614,"2,220.10","1,363,142.09","741,382.49","621,759.61"
35,"Mountain-200 Silver, 38",596,"2,247.42","1,339,462.79","728,598.36","610,864.43"
37,"Mountain-200 Silver, 46",580,"2,243.28","1,301,100.10","707,609.63","593,490.47"
32,"Mountain-200 Black, 38",582,"2,224.86","1,294,866.14","704,388.68","590,477.46"
36,"Mountain-200 Silver, 42",560,"2,245.42","1,257,434.57","683,922.64","573,511.94"
58,"Road-150 Red, 48",337,"3,578.27","1,205,876.99","731,726.15","474,150.84"
61,"Road-150 Red, 62",336,"3,578.27","1,202,298.72","729,554.85","472,743.87"
59,"Road-150 Red, 52",302,"3,578.27","1,080,637.54","655,730.85","424,906.69"
60,"Road-150 Red, 56",295,"3,578.27","1,055,589.65","640,531.79","415,057.86"


The product with the most sales is Mountain-200 Black, 46, with an average price of $2,215.27.


In [ ]:
best_product = top_product['EnglishProductName']

bike_sales = df[
    df["EnglishProductName"] == best_product
]

yearly_bike_sales = bike_sales.groupby("OrderYear")["SalesAmount"].sum()

print(yearly_bike_sales.apply(lambda amount: f"${amount:,.2f}"))

OrderYear
2011      $2,049.10
2012    $423,589.58
2013    $947,830.87
Name: SalesAmount, dtype: object


In [ ]:
# Add up sales and calculate the average unit price for each category.
# Sort total sales from highest to lowest so the highest-selling category appears first with ascending=False.

category_summary = df.groupby('ProductCategory').agg(
    Total_Sales=('SalesAmount', 'sum'),
    Average_Price=('UnitPrice', 'mean')
).sort_values('Total_Sales', ascending=False)


display(category_summary.style.format('{:,.2f}'))
print(f"The category with the highest total sales is {category_summary.index[0]}, with an average price of ${category_summary.iloc[0]['Average_Price']:,.2f}.")

,Total_Sales,Average_Price
ProductCategory,,
Bikes,"28,318,144.65","1,862.42"
Accessories,"700,759.96",19.42
Clothing,"339,772.61",37.33


The category with the highest total sales is Bikes, with an average price of $1,862.42.


In [ ]:
# Sort total sales from highest to lowest so the highest-selling category appears first with ascending=True.

category_summary = df.groupby('ProductCategory').agg(
    Total_Sales=('SalesAmount', 'sum'),
    Average_Price=('UnitPrice', 'mean')
).sort_values('Total_Sales', ascending=True)

display(category_summary.style.format('{:,.2f}'))
print(f"The category with the lowest total sales is {category_summary.index[0]}, with an average price of ${category_summary.iloc[0]['Average_Price']:,.2f}.")

,Total_Sales,Average_Price
ProductCategory,,
Clothing,"339,772.61",37.33
Accessories,"700,759.96",19.42
Bikes,"28,318,144.65","1,862.42"


The category with the lowest total sales is Clothing, with an average price of $37.33.


Does owning a home make a difference in sales?

In [ ]:
pd.options.display.float_format = '{:,.2f}'.format

df.groupby('HouseOwnerFlag')['SalesAmount'].agg(['sum', 'mean', 'count'])

,sum,mean,count
HouseOwnerFlag,,,
0,"8,973,327.42",479.88,18699
1,"20,385,349.80",488.87,41699
